# Lahore NDVI (Sentinel-2 L2A) — Oct 2024
# Granularity: 10 m | Outputs: GeoTIFF + UC-wise CSV


### 0. Initialize Earth Engine 


In [32]:
import ee, geemap, geopandas as gpd, pandas as pd

ee.Authenticate()
ee.Initialize()

In [33]:
UC_SHP = "../../data/Lahore UCs/Lahore UC.shp"   # any Lahore boundary works; UC union is fine
START, END = "2024-10-01", "2024-10-31"
S2 = "COPERNICUS/S2_SR_HARMONIZED"
SAMPLE_SCALE = 100      # <-- 100 m keeps filesize sane. Avoid <50 m for whole Lahore.
SMOOTH_RADIUS_M = 0     # e.g., 30 or 50 to gently smooth NDVI (0 = no smoothing)
OUT_PREFIX = "NDVI_Lahore_Oct2024"

In [34]:
gdf = gpd.read_file(UC_SHP)
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
gdf["geometry"] = gdf["geometry"].buffer(0)
ucs_fc = geemap.gdf_to_ee(gdf)
region = ucs_fc.geometry()

# ---------- 2) Sentinel-2 NDVI (10 m), cloud-masked ----------
def mask_s2_sr(img):
    qa = img.select("QA60")
    cloud  = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus = qa.bitwiseAnd(1 << 11).eq(0)
    return img.updateMask(cloud.And(cirrus))

def add_ndvi(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return img.addBands(ndvi)

col = (ee.ImageCollection(S2)
       .filterBounds(region)
       .filterDate(START, END)
       .map(mask_s2_sr)
       .map(add_ndvi)
       .select("NDVI"))

ndvi_med = col.median().rename("NDVI")

# Optional gentle spatial smoothing (interpolation-look), in meters
if SMOOTH_RADIUS_M > 0:
    kernel = ee.Kernel.circle(radius=SMOOTH_RADIUS_M, units='meters', normalize=True)
    ndvi_med = ndvi_med.focal_mean(kernel=kernel, iterations=1)

# ---------- 3) Export POINT GRID (~SAMPLE_SCALE) ----------
# One feature per sample cell with lat/lon + NDVI value "val"
pts_fc = ee.Image.pixelLonLat().addBands(ndvi_med.rename("val")).sample(
    region=region,
    scale=SAMPLE_SCALE,
    geometries=True,
    seed=1
)

# GeoJSON (best for Folium, QGIS)
geemap.ee_export_vector(pts_fc, filename=f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m.geojson")
print(f"[OK] Saved → {OUT_PREFIX}_points_{SAMPLE_SCALE}m.geojson")

Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/NDVI/NDVI_Lahore_Oct2024_points_100m.geojson
[OK] Saved → NDVI_Lahore_Oct2024_points_100m.geojson


In [27]:
# (optional) CSV for Folium HeatMap (lighter to parse)
df_pts = geemap.ee_to_df(pts_fc)
# harmonize columns
df_pts = df_pts.rename(columns={"latitude":"lat","longitude":"lon"})
df_pts[["lat","lon","val"]].to_csv(f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m.csv", index=False)
print(f"[OK] Saved → {OUT_PREFIX}_points_{SAMPLE_SCALE}m.csv")

[OK] Saved → NDVI_Lahore_Oct2024_points_100m.csv


In [28]:
import pandas as pd, folium
from folium.plugins import HeatMap

dfp = pd.read_csv(f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m.csv").dropna(subset=["lat","lon","val"])
# robust weights 10–90% to avoid outliers
vmin, vmax = dfp["val"].quantile(0.10), dfp["val"].quantile(0.90)
if vmin == vmax: vmax = vmin + 1e-6
dfp["w"] = ((dfp["val"] - vmin) / (vmax - vmin)).clip(0,1)

m = folium.Map(location=[31.5204,74.3587], zoom_start=11, tiles="cartodbpositron", control_scale=True)
HeatMap(dfp[["lat","lon","w"]].values.tolist(), radius=18, blur=28, min_opacity=0.3, name="NDVI heat").add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(f"{OUT_PREFIX}_heatmap.html")
m
print(f"[OK] Saved → {OUT_PREFIX}_heatmap.html")

[OK] Saved → NDVI_Lahore_Oct2024_heatmap.html


In [35]:
import os

if df_pts.empty:
    raise ValueError("df_pts is empty; export point samples before creating the shapefile.")

points_gdf = gpd.GeoDataFrame(
    df_pts.copy(),
    geometry=gpd.points_from_xy(df_pts["lon"], df_pts["lat"]),
    crs="EPSG:4326"
)

shp_dir = f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m_shp"
os.makedirs(shp_dir, exist_ok=True)

base_name = f"{OUT_PREFIX}_points_{SAMPLE_SCALE}m"
for ext in (".shp", ".shx", ".dbf", ".prj", ".cpg"):
    existing = os.path.join(shp_dir, f"{base_name}{ext}")
    if os.path.exists(existing):
        os.remove(existing)

shp_path = os.path.join(shp_dir, f"{base_name}.shp")
points_gdf.to_file(shp_path, driver="ESRI Shapefile")
print(f"[OK] Saved Shapefile set → {shp_dir}")
print("Files generated:")
print(" ", "\n  ".join(sorted(os.listdir(shp_dir))))


[OK] Saved Shapefile set → NDVI_Lahore_Oct2024_points_100m_shp
Files generated:
  NDVI_Lahore_Oct2024_points_100m.cpg
  NDVI_Lahore_Oct2024_points_100m.dbf
  NDVI_Lahore_Oct2024_points_100m.prj
  NDVI_Lahore_Oct2024_points_100m.shp
  NDVI_Lahore_Oct2024_points_100m.shx
